In [1]:
# ── Ячейка 1: импорты и инфраструктура ────────────────────────────────────────
import sys; sys.path.insert(0, "..")
import gc, time
import numpy as np
import joblib

from src import config, data, evaluate

con = data.get_duckdb_connection()
feature_cols = data.get_feature_cols(con)   # assert: 76 признаков, таргет не утёк

n_train = data.count_rows(con, config.TRAIN_PATH)
n_val   = data.count_rows(con, config.VAL_PATH)
n_test  = data.count_rows(con, config.TEST_PATH)

print(f"Признаков (tree-view): {len(feature_cols)}")
print(f"train={n_train:,}  val={n_val:,}  test={n_test:,}")

Признаков (tree-view): 76
train=9,583,883  val=2,053,688  test=2,053,697


In [2]:
# ── Ячейка 2: загрузка train в RAM (tree-view, сырьё) ─────────────────────────
# RF инвариантен к масштабу и принимает -1 как есть — препроцессор НЕ нужен.
# Грузим сырые 76 признаков целиком (как в старом ноутбуке, ~2.46 GB).
print("Загружаем train (сырьё, 76 признаков)...")
t0 = time.time()

X_chunks, y_chunks = [], []
for X_b, y_b in data.iter_batches(config.TRAIN_PATH, feature_cols):
    X_chunks.append(X_b)
    y_chunks.append(y_b)

X_train = np.concatenate(X_chunks); del X_chunks
y_train = np.concatenate(y_chunks); del y_chunks
gc.collect()

print(f"X_train: {X_train.shape}  {X_train.nbytes/1024**3:.2f} GB  "
      f"malicious={y_train.mean():.4f}  |  {time.time()-t0:.1f}s")

Загружаем train (сырьё, 76 признаков)...
X_train: (9583883, 76)  2.71 GB  malicious=0.2696  |  4.7s


In [3]:
# ── Ячейка 3: обучение RandomForest ───────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    class_weight="balanced",
    n_jobs=-1,
    random_state=config.RANDOM_STATE,
    verbose=1,
)

print("Обучение RandomForest (100 деревьев, max_depth=None)...")
t0 = time.time()
model.fit(X_train, y_train)
print(f"\nГотово за {time.time()-t0:.1f}s")

del X_train, y_train; gc.collect()

joblib.dump(model, config.MODELS_DIR / "rf.pkl")
print(f"Модель сохранена → {config.MODELS_DIR / 'rf.pkl'}")

Обучение RandomForest (100 деревьев, max_depth=None)...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:  4.7min
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed: 16.7min finished



Готово за 1006.6s
Модель сохранена → /home/kirill/Рабочий стол/Диплом/traffic-classification-vkr_v2.0/models/rf.pkl


In [4]:
# ── Ячейка 4: оценка на VAL (подбор порога) ───────────────────────────────────
# tree-view: предикт на сыром батче, БЕЗ препроцессора.
def predict_rf(X_raw):
    return model.predict_proba(X_raw)[:, 1]

m_val = evaluate.evaluate(
    "rf", predict_rf, config.VAL_PATH, feature_cols,
    threshold=None, split_name="val",
)
best_thr = m_val["threshold"]
print(f"\nПодобранный на val порог: {best_thr}")

[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.2s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    1.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.2s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    1.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.2s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    1.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.2s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    1.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0


  rf  (val)
строк 2,053,688 за 5.93s  |  порог 0.39 (tuned_on_this_split)
F1_macro 0.9988  MCC 0.9977  PR-AUC 0.999539  ROC-AUC 0.999908  FPR@95 0.000146
TN=1,498,116  FP=1,884  FN=10  TP=553,678  (FPR=0.0013, FNR=0.0000)

per-class recall:
  Benign                           n=1,500,000  recall=99.87%
  DoS Hulk                         n= 270,445  recall=100.00%
  DDoS HOIC                        n= 161,157  recall=100.00%
  DDoS LOIC-HTTP                   n=  43,399  recall=100.00%
  FTP-Patator                      n=  28,545  recall=100.00%
  DoS Slowhttptest                 n=  15,832  recall=100.00%
  Bot                              n=  14,423  recall=99.99%
  SSH-Patator                      n=  13,897  recall=100.00%
  DoS GoldenEye                    n=   4,029  recall=100.00%
  DoS Slowloris                    n=   1,541  recall=99.55%
  DDoS LOIC-UDP                    n=     357  recall=100.00%
  Web Attack - Brute Force         n=      39  recall=97.44%
  Web Attack - XS

In [5]:
# ── Ячейка 5: оценка на TEST (замороженный порог) ─────────────────────────────
m_test = evaluate.evaluate(
    "rf", predict_rf, config.TEST_PATH, feature_cols,
    threshold=best_thr, split_name="test",
)

[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.2s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    1.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.3s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    1.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.2s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    1.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.2s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    1.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0


  rf  (test)
строк 2,053,697 за 5.74s  |  порог 0.39 (frozen_from_val)
F1_macro 0.9985  MCC 0.997  PR-AUC 0.998566  ROC-AUC 0.999729  FPR@95 0.000489
TN=1,497,602  FP=2,398  FN=12  TP=553,685  (FPR=0.0016, FNR=0.0000)

per-class recall:
  Benign                           n=1,500,000  recall=99.84%
  DoS Hulk                         n= 270,445  recall=100.00%
  DDoS HOIC                        n= 161,157  recall=100.00%
  DDoS LOIC-HTTP                   n=  43,400  recall=100.00%
  FTP-Patator                      n=  28,545  recall=100.00%
  DoS Slowhttptest                 n=  15,833  recall=100.00%
  Bot                              n=  14,424  recall=100.00%
  SSH-Patator                      n=  13,898  recall=100.00%
  DoS GoldenEye                    n=   4,030  recall=100.00%
  DoS Slowloris                    n=   1,542  recall=99.35%
  DDoS LOIC-UDP                    n=     358  recall=100.00%
  Web Attack - Brute Force         n=      39  recall=97.44%
  Web Attack - XSS  

In [6]:
# ── Ячейка 6: feature importance (для записки) ────────────────────────────────
# Не влияет на метрики — диагностика для отчёта: на какие признаки опирается RF.
import pandas as pd

fi = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

print("Топ-15 признаков по важности (RF):")
print(fi.head(15).to_string(index=False))
print(f"\nТоп-3 суммарно: {fi['importance'].head(3).sum():.4f}")

Топ-15 признаков по важности (RF):
               feature  importance
       bwd_pkt_len_std    0.074738
fwd_tcp_init_win_bytes    0.072378
         down_up_ratio    0.064396
           pkt_len_max    0.052861
               iat_min    0.049844
           pkt_len_var    0.039550
           pkt_len_std    0.039168
   fwd_pkt_hdr_len_min    0.037362
       bwd_pkt_len_max    0.035835
      bwd_pkt_len_mean    0.034045
bwd_tcp_init_win_bytes    0.032027
       bwd_pkt_len_tot    0.028541
   bwd_pkt_hdr_len_tot    0.027611
           fwd_iat_min    0.024180
              flag_fin    0.023798

Топ-3 суммарно: 0.2115
